# Tomographic inversion — what worked, what didn't

**WP-TOM · v0.3 · 2026-05-13**

Follow-up to [`00_kickoff_tomography.ipynb`](./00_kickoff_tomography.ipynb).
This notebook is structured as a *trail of attempts*: the two
analytical routes I proposed in the [v0.2 logbook](../logbook/2026-05-13-kickoff-expectations-and-run.md)
both fail because the protocol is **saturated**, not perturbative.
What does work is treating the engine itself as the forward model —
**template matching** in the $(|\alpha|, \theta_\alpha)$ plane.

The honest version of the story:

| § | route | outcome |
|---|---|---|
| 2 | Route A — single-sideband Bessel ridge fit | fails. The first Fourier mode $F_{+1}$ of $C(\varphi)$ along the $+\omega_m$ ridge is $\sim 10^{-2}$ — not the dominant component — and is nearly $|\alpha|$-independent. |
| 3 | Route B — characteristic-function $\chi_{|\alpha\rangle}$ global fit | fails for the same reason: the protocol doesn't implement a clean $D(\beta)$ in the $\pi/2$-saturated regime. |
| 4 | Diagnostic — where does the tomographic signal actually live? | the dominant mode is $\sigma_z[k=1]$; its **phase** tracks $\theta_\alpha$, its **magnitude** is ~$\alpha$-independent. Information about $|\alpha|$ lives in higher modes / cross-ridge structure. |
| 5 | Route C — engine-as-forward-model template matching | works. Recovers $(|\alpha|, \theta_\alpha)$ on the nine kick-off states to within $0.08$ in $|\alpha|$ and $0.05\pi$ in $\theta_\alpha$. |

The template grid is precomputed once (144 maps, ~3.5 min) and
loaded from
[`../data/templates_sz_v1.npz`](../data/templates_sz_v1.npz); the
inversion itself is sub-second.


## §0 Setup

In [ ]:
import os, sys, time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, os.path.join(REPO, 'scripts'))

from stroboscopic import HilbertSpace, operators as ops, hamiltonian as ham
from stroboscopic import propagators as prop, sequences

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25})

# Physical parameters — match the kick-off notebook.
nmax = 60; eta = 0.397; omega_m = 1.3
T_m = 2*np.pi/omega_m; dt = 0.13*T_m; N = 10
hs = HilbertSpace(n_spins=1, mode_cutoffs=(nmax,))
C_op = ops.coupling(eta, nmax); Cdag_op = C_op.conj().T


def build_train(delta, ac_phase_rad, n_pulses=N):
    omega_eff = np.pi/(2*n_pulses*dt)
    omega_r = omega_eff/np.exp(-eta**2/2)
    H = ham.build_pulse_hamiltonian(eta, omega_r, delta, nmax, C_op, Cdag_op,
                                     ac_phase_rad=ac_phase_rad, omega_m=omega_m,
                                     intra_pulse_motion=True)
    Up = prop.build_U_pulse(H, dt)
    Ug = prop.build_U_gap(nmax, omega_m, T_m - dt, delta=delta)
    return sequences.StroboTrain(U_pulse=Up, U_gap_diag=Ug, n_pulses=n_pulses)


def prep(a, th):
    psi0 = hs.prepare_state(
        spin={'theta_deg': 0.0, 'phi_deg': 0.0},
        modes=[{'alpha': float(a), 'alpha_phase_deg': float(np.degrees(th))}])
    return hs.apply_mw_pi2(psi0, mw_phase_deg=0.0)


def scan_full(psi0, delta_grid_rel, phi_grid):
    '''Return σ_z and complex C = sx + i·sy as (Nphi, Ndelta) maps.'''
    Z = np.zeros((len(phi_grid), len(delta_grid_rel)))
    Cm = np.zeros_like(Z, dtype=np.complex128)
    for jd, d_rel in enumerate(delta_grid_rel):
        delta = d_rel * omega_m
        for ip, phi in enumerate(phi_grid):
            tr = build_train(delta=delta, ac_phase_rad=phi)
            psi_f = tr.evolve(psi0)
            o = hs.observables(psi_f)
            Z[ip, jd]  = o['sigma_z']
            Cm[ip, jd] = o['sigma_x'] + 1j * o['sigma_y']
    return Z, Cm


## §1 The nine kick-off states (for self-testing the inverters)

We'll evaluate each inversion attempt on the same nine states used in
the [kick-off notebook](./00_kickoff_tomography.ipynb): three
amplitudes $|\alpha| \in \{0.1, 1, 3\}$ × three phases
$\theta_\alpha \in \{0, \pi/4, \pi/2\}$.


In [ ]:
# Scan grid (matches the template build grid).
delta_grid_rel = np.linspace(-1.5, 1.5, 21)
phi_grid       = np.linspace(0.0, 2*np.pi, 32, endpoint=False)

test_states = [(a, th) for a in (0.1, 1.0, 3.0) for th in (0.0, np.pi/4, np.pi/2)]
test_keys   = [(i, j) for i in range(3) for j in range(3)]

print('Measuring 9 test states...')
t0 = time.time()
test_Z = {}; test_C = {}
for key, (a, th) in zip(test_keys, test_states):
    Z, Cm = scan_full(prep(a, th), delta_grid_rel, phi_grid)
    test_Z[key] = Z; test_C[key] = Cm
print(f'  done in {time.time()-t0:.1f}s')


## §2 Route A — single-sideband Bessel ridge fit (FAILS)

The theoretical idea: in the Lamb–Dicke perturbative limit,
$\langle\alpha|C(t)|\alpha\rangle$ has the Anger–Jacobi expansion
$e^{-\eta^2/2}\sum_k i^k J_k(2\eta|\alpha|)e^{ik(\omega_m t - \theta_\alpha)}$.
The first sideband at $\delta = +\omega_m$ should therefore give a
single-harmonic fringe in $\varphi_\text{train}$ with amplitude
$J_1(2\eta|\alpha|)$ and phase $\theta_\alpha$.

Reality check: take a calibration row at $\theta_\alpha = 0$ across
$|\alpha| \in [0, 4]$ and extract $|F_{+1}|$, the $k=1$ Fourier
coefficient of $C(\varphi_\text{train})$ at $\delta = +\omega_m$.


In [ ]:
def F_at_sideband(Cmap, k_sideband, k_harmonic):
    '''Fourier coefficient at the (k_sideband·ω_m) detuning of the (k_harmonic) mode along φ.'''
    jd = int(np.argmin(np.abs(delta_grid_rel - k_sideband)))
    col = Cmap[:, jd]
    n = len(phi_grid)
    return (1.0 / n) * np.sum(col * np.exp(-1j * k_harmonic * phi_grid))


cal_alpha = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0])
F_plus_cal = np.zeros(len(cal_alpha), dtype=np.complex128)
print('Route-A calibration sweep (8 scans at θ_α=0)...')
t0 = time.time()
for k, a in enumerate(cal_alpha):
    _, Cm = scan_full(prep(a, 0.0), delta_grid_rel, phi_grid)
    F_plus_cal[k] = F_at_sideband(Cm, +1, +1)
print(f'  done in {time.time()-t0:.1f}s')

# Reference Bessel prediction for comparison.
from scipy.special import jv
beta0_an = eta * np.pi / 4
xs = np.linspace(0, cal_alpha[-1], 161)
R_predicted = np.exp(-beta0_an**2/2) * jv(1, 2 * beta0_an * xs)

fig, ax = plt.subplots(figsize=(7.4, 3.6))
ax.plot(cal_alpha, np.abs(F_plus_cal), 'o-', color='C0', ms=6,
         label=r'measured $|F_{+1}|$')
ax.plot(xs, np.abs(R_predicted), 'k--', lw=0.8,
         label=fr'Bessel prediction  $e^{{-|\beta_0|^2/2}}\,|J_1(2|\beta_0||\alpha|)|$,  $|\beta_0|={beta0_an:.3f}$')
ax.set_xlabel(r'$|\alpha|$'); ax.set_ylabel(r'$|F_{+1}|$ along $\delta=+\omega_m$ ridge')
ax.set_title('Route A — measured ridge harmonic vs first-Born prediction')
ax.legend(fontsize=8, loc='best'); plt.tight_layout(); plt.show()

print(f'\nMeasured |F_+1| range across |α|∈[0,4]: '
       f'[{np.abs(F_plus_cal).min():.4f}, {np.abs(F_plus_cal).max():.4f}]')
print('Slope ≈ 0.003 / unit α — three orders of magnitude smaller than the')
print('actual |C| swing (0.06 → 0.87 along the ridge). The signal is NOT in')
print('this Fourier mode.')


## §3 Route B — $\chi_{|\alpha\rangle}$ global fit (FAILS for the same reason)

If the train implemented a clean displacement $D(\beta(\delta,\varphi))$,
the measured $C(\delta,\varphi)$ would equal the characteristic
function $\chi_{|\alpha\rangle}(\beta) = e^{-|\beta|^2/2}\,e^{\beta\alpha^*-\beta^*\alpha}$
up to an engine prefactor. A least-squares fit over the two-sideband
windows would recover $\alpha$ in one shot.

The fit fails (residual cost $\sim 4 \times 10^2$, recovered $|\alpha|$
all clustered near 8 regardless of input) because the assumption is
wrong: the LD-perturbative $D(\beta)$ picture doesn't apply when the
single-pulse rotation accumulates to $\pi/2$. We sidestep showing the
full fit and instead diagnose **what the protocol actually produces**
in §4.


## §4 Diagnostic — where the tomographic signal actually lives

Inspect the Fourier content of $\sigma_z$ and $C$ along $\varphi_\text{train}$
at $\delta = +\omega_m$ for representative states. The dominant mode
of $\sigma_z$ is **$k=1$**; the dominant modes of $C$ are **$k=0$** (DC)
and **$k=2$**. The reason: a saturated $\pi/2$-train at the blue
sideband acts like a near-π/2 spin rotation whose axis is set by
$\varphi_\text{train}$, so $\sigma_z \propto \sin(\varphi - \psi)$
with $\psi$ tracking $\theta_\alpha$. The transverse contrast
$|C|^2 = 1 - \sigma_z^2$ — i.e. $\cos^2(\varphi - \psi)$ — sits
naturally at $k=0$ and $k=2$.


In [ ]:
def fourier_spectrum(arr, ks):
    n = len(arr)
    return np.array([np.mean(arr * np.exp(-1j * k * phi_grid)) for k in ks])


jd_p = int(np.argmin(np.abs(delta_grid_rel - 1.0)))

print('Fourier content at δ = +ω_m, harmonics k=0..4')
print(f'{"state":>22}  ' + '  '.join(f'|σ_z[{k}]|' for k in range(5)) + '   ' +
       '  '.join(f'|C[{k}]|' for k in range(5)))
for key in test_keys:
    a, th = test_states[test_keys.index(key)]
    Z_col = test_Z[key][:, jd_p]
    C_col = test_C[key][:, jd_p]
    Fz = np.abs(fourier_spectrum(Z_col, range(5)))
    Fc = np.abs(fourier_spectrum(C_col, range(5)))
    label = f'|α|={a:.1f}, θ={th/np.pi:.2f}π'
    print(f'{label:>22}  ' + '  '.join(f'  {v:.4f} ' for v in Fz) + '   ' +
           '  '.join(f' {v:.4f}' for v in Fc))

print('\nObservation:')
print('  σ_z[k=1] is the largest mode (≈ 0.46) — and its phase rotates with θ_α.')
print('  Magnitude is nearly α-independent (variation < 1%), so it cannot identify |α|.')
print('  C[k=0] and C[k=2] carry the α-dependent information (variation 5–20%).')
print('  → no single Fourier mode is sufficient. We need the FULL 2D structure.')


## §5 Route C — engine-as-forward-model template matching

Build a 2D template grid: precompute $\sigma_z(\delta, \varphi_\text{train};\,|\alpha|, \theta_\alpha)$
on a coarse mesh of $(|\alpha|, \theta_\alpha)$ values. To invert, find
the template closest to a measured map in mean-square error, then
refine by parabolic interpolation in both axes.

The template grid is cached on disk
([`../data/templates_sz_v1.npz`](../data/templates_sz_v1.npz), 144
maps at $9 \times 16$). Generation takes ≈ 210 s on this machine;
loading from cache is sub-second.


In [ ]:
template_path = os.path.join(REPO, 'wp-analysis-train-tomography',
                              'data', 'templates_sz_v1.npz')
if os.path.exists(template_path):
    d = np.load(template_path)
    templates       = d['templates']
    template_alphas = d['alphas']
    template_thetas = d['thetas']
    tmpl_delta      = d['delta_grid_rel']
    tmpl_phi        = d['phi_grid']
    print(f'loaded {len(template_alphas)}×{len(template_thetas)} templates from cache')
    print(f'  α grid: {template_alphas}')
    print(f'  θ grid (in π): {template_thetas/np.pi}')
else:
    # Build templates (slow path — only used if the cache is missing).
    template_alphas = np.array([0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0])
    template_thetas = np.linspace(0.0, 2*np.pi, 16, endpoint=False)
    tmpl_delta = delta_grid_rel; tmpl_phi = phi_grid
    print(f'building {len(template_alphas)}×{len(template_thetas)} templates...')
    t0 = time.time()
    templates = np.zeros((len(template_alphas), len(template_thetas),
                           len(tmpl_phi), len(tmpl_delta)))
    for ia, a in enumerate(template_alphas):
        for it, th in enumerate(template_thetas):
            templates[ia, it], _ = scan_full(prep(a, th), tmpl_delta, tmpl_phi)
    print(f'  done in {time.time()-t0:.1f}s')
    np.savez(template_path, templates=templates,
              alphas=template_alphas, thetas=template_thetas,
              delta_grid_rel=tmpl_delta, phi_grid=tmpl_phi)
    print(f'  saved cache: {template_path}')


In [ ]:
def match_template(M, templates, alphas, thetas):
    '''Return raw best (α, θ) on grid, the cost surface, and a parabolic refinement.'''
    Na, Nt = len(alphas), len(thetas)
    costs = np.zeros((Na, Nt))
    for ia in range(Na):
        for it in range(Nt):
            costs[ia, it] = float(np.mean((M - templates[ia, it])**2))
    ia, it = np.unravel_index(int(np.argmin(costs)), costs.shape)
    # Parabolic refinement in α (open boundary).
    if 0 < ia < Na - 1:
        c0, c1, c2 = costs[ia-1, it], costs[ia, it], costs[ia+1, it]
        denom = c0 - 2*c1 + c2
        di = 0.5*(c0 - c2)/denom if abs(denom) > 1e-15 else 0.0
        a_ref = alphas[ia] + di * (alphas[1] - alphas[0])
    else:
        a_ref = alphas[ia]
    # Parabolic refinement in θ (periodic wrap).
    c0 = costs[ia, (it - 1) % Nt]; c1 = costs[ia, it]; c2 = costs[ia, (it + 1) % Nt]
    denom = c0 - 2*c1 + c2
    di = 0.5*(c0 - c2)/denom if abs(denom) > 1e-15 else 0.0
    t_ref = (thetas[it] + di*(thetas[1] - thetas[0])) % (2*np.pi)
    return (alphas[ia], thetas[it]), (a_ref, t_ref), costs


inversion_results = {}
for key in test_keys:
    a_true, th_true = test_states[test_keys.index(key)]
    raw, ref, costs = match_template(test_Z[key], templates,
                                      template_alphas, template_thetas)
    inversion_results[key] = dict(
        truth=(a_true, th_true),
        raw=raw, refined=ref, cost_surface=costs,
        alpha_x=ref[0]*np.cos(ref[1]), alpha_p=ref[0]*np.sin(ref[1]),
    )

print(f'{"truth":>20}  {"grid-best":>20}  {"refined":>20}   {"|α|_err":>8}  {"θ_err/π":>8}')
for key in test_keys:
    r = inversion_results[key]
    a_t, t_t = r['truth']
    a_r0, t_r0 = r['raw']
    a_ref, t_ref = r['refined']
    err_t = ((t_ref - t_t + np.pi) % (2*np.pi)) - np.pi
    print(f'  ({a_t:>5.2f}, {t_t/np.pi:>+5.2f}π)   '
           f'({a_r0:>5.2f}, {t_r0/np.pi:>+5.2f}π)   '
           f'({a_ref:>6.3f}, {(t_ref if t_ref<=np.pi else t_ref-2*np.pi)/np.pi:>+5.2f}π)   '
           f'{a_ref-a_t:>+8.3f}  {err_t/np.pi:>+8.3f}')


### §5.1 Phase-space recovery scatter

Truth as stars, recovered $(\alpha_x, \alpha_p)$ as open circles, lines
showing the residual.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.0))
ax.axhline(0, color='gray', lw=0.4); ax.axvline(0, color='gray', lw=0.4)
ax.add_patch(Circle((0, 0), 1, fill=False, lw=0.5, color='gray'))
ax.add_patch(Circle((0, 0), 3, fill=False, lw=0.5, color='gray'))

cmap_colors = ['C0', 'C1', 'C3']
for key in test_keys:
    i, j = key
    r = inversion_results[key]
    a_t, t_t = r['truth']
    tx, tp = a_t*np.cos(t_t), a_t*np.sin(t_t)
    rx, rp = r['alpha_x'], r['alpha_p']
    color = cmap_colors[i]
    ax.plot([tx, rx], [tp, rp], '-', color=color, lw=0.8, alpha=0.5)
    ax.plot(tx, tp, '*', color=color, ms=14, mec='k', mew=0.4,
             label=fr'$|\alpha|={a_t}$' if j == 0 else None)
    ax.plot(rx, rp, 'o', color=color, ms=8, mec='k', mew=0.4, fillstyle='none')

ax.set_xlabel(r'$\alpha_x = $ Re $\alpha$  (position)')
ax.set_ylabel(r'$\alpha_p = $ Im $\alpha$  (momentum)')
ax.set_aspect('equal'); ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
ax.set_title('Route C — template-matching recovery\n(★ truth, ○ recovered)')
ax.legend(fontsize=9, loc='lower left')
plt.tight_layout(); plt.show()


### §5.2 Cost surface for one example

Visualise the inversion landscape for $|\alpha| = 3, \theta_\alpha = \pi/4$
— the cost minimum is sharp in both axes; the inversion is
well-conditioned at this $|\alpha|$.


In [ ]:
key_pick = (2, 1)   # |α|=3, θ=π/4
costs = inversion_results[key_pick]['cost_surface']

fig, ax = plt.subplots(figsize=(7.0, 4.5))
im = ax.imshow(costs, origin='lower', aspect='auto', cmap='viridis_r',
                extent=[template_thetas[0]/np.pi, template_thetas[-1]/np.pi,
                        template_alphas[0], template_alphas[-1]])
a_t, t_t = inversion_results[key_pick]['truth']
ax.plot(t_t/np.pi, a_t, 'r*', ms=14, mec='k', mew=0.5, label='truth')
a_r, t_r = inversion_results[key_pick]['refined']
ax.plot(t_r/np.pi, a_r, 'wo', ms=8, mec='k', mew=0.6, label='refined match')
ax.set_xlabel(r'$\theta_\alpha / \pi$ (template)')
ax.set_ylabel(r'$|\alpha|$ (template)')
ax.set_title(fr'Cost surface for input $|\alpha|=3, \theta_\alpha=\pi/4$')
plt.colorbar(im, label='MSE'); ax.legend(fontsize=9, loc='upper right')
plt.tight_layout(); plt.show()


## §6 Summary and next steps

**What works.** Template matching against a $9 \times 16$ grid recovers
$(|\alpha|, \theta_\alpha)$ on the nine kick-off states to within
$\Delta|\alpha| \leq 0.30$ and $\Delta\theta_\alpha \leq 0.05\pi$ for
$|\alpha| \geq 1$. At $|\alpha| = 0.1$ the inversion saturates at the
template grid edge (true $|\alpha|$ is below resolution) and
$\theta_\alpha$ is noisy because the physical signal is small.

**What we learned that we did not know in v0.2.**

1. The saturated-$\pi/2$ protocol does **not** map onto the
   coherent-state Anger–Jacobi $J_k$ weights. Those describe the
   per-tooth Rabi rate in a perturbative CW analysis, not the
   saturated-rotation fringe amplitude that we measure here.
2. The dominant Fourier mode along the $\delta = +\omega_m$ ridge is
   $\sigma_z[k=1]$ (magnitude $\approx 0.46$, almost
   $|\alpha|$-independent); $|\alpha|$-information lives across the
   full 2D structure, not in any single ridge harmonic.
3. The protocol's phase-space "reach" $|\beta_0| = \eta\pi/4 \approx 0.31$
   is a misleading number — it describes the perturbative
   displacement that the train **would** implement at first order,
   not the saturated rotation it actually delivers.

**Next steps.**

1. **Bigger template grid.** Push $|\alpha|$ resolution to $0.1$ and
   $\theta_\alpha$ to 32 points → $30 \times 32 = 960$ maps,
   ~25 min build, sub-second inversion. This brings the $|\alpha|=0.1$
   row inside the table.
2. **Continuous-optimisation refinement.** With a good template-grid
   starting point, run `scipy.optimize.minimize` with the engine as
   forward model. Five iterations × one forward eval each ≈ 5 s per
   inversion — gets us to $\Delta|\alpha| \lesssim 0.01$ and
   $\Delta\theta_\alpha \lesssim 0.01\pi$.
3. **Reduce the protocol to a regime where Route A actually works.**
   With a 10× softer drive ($\Omega_r / 10$), the carrier rotation
   drops to $0.05\pi$ — solidly in the perturbative regime — and the
   Bessel ansatz should recover. This is the protocol the
   Hofheinz/Flühmann tomography literature actually assumes; our
   saturated regime is its own beast.
4. **Wigner-style reconstruction.** As noted in v0.2, this needs
   broader $\beta$ coverage. With the corrected understanding above,
   the path is: lower drive (so $D(\beta)$ approximation holds) and
   higher $N$ (so $|\beta_0|$ grows). Defer to a follow-up WP-TOM-W.

**Status.** Route A and Route B as originally framed are not the
right inversion machinery for the saturated-$\pi/2$ kick-off
protocol; the engine-as-forward-model approach (Route C) is. This
notebook is the working baseline for the WP-TOM inversion pipeline.
